# Spurious regression, unit roots, and cointegration

This notebook is an executable companion to [the article](article.md). Every series below is simulated. The first exercise builds unrelated random walks; the second constructs cointegration by design; the third exposes the distinct test statistics; the last regenerates smoke-test evidence in a temporary directory. Run all cells in a clean Python 3.12 kernel after installing `requirements.txt`.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np

from analysis import (Design, cointegrated_pair, dickey_fuller_t, ecm_fit,
                      engle_granger_t, independent_walks, ols_levels, run_study)

design = Design()
print(f'Seed: {design.seed}; horizon: {design.horizon}')

## 1. An apparently convincing false relationship

Two independent Gaussian innovations are accumulated independently. A *conventional* levels-regression t-test then uses an invalid stationary-IID reference distribution. Its p-value is deliberately naive. Differencing recovers the original independent increments **for this exact generating process**, not as a generic causal remedy.

In [ ]:
random = np.random.default_rng(42)
x, y = independent_walks(250, random)
levels = ols_levels(y, x)
increments = ols_levels(np.diff(y), np.diff(x))
assert np.array_equal(np.diff(x), x[1:] - x[:-1])
print({'levels_r2': levels.r_squared, 'levels_naive_p': levels.p_naive,
       'differences_r2': increments.r_squared, 'differences_naive_p': increments.p_naive})

## 2. Construct a genuine cointegrating relation

Let `X` be a Gaussian random walk, `U` a stationary AR(1), and `Y = 1.6 X + U`. Then the **known true** linear combination `Y - 1.6 X` equals `U`. Because `U_t = 0.65 U_(t-1) + xi_t`, the exact identity is `Delta Y = 1.6 Delta X - 0.35 U_(t-1) + xi_t`. The fitted error-correction term uses an estimated, not observed, residual.

In [ ]:
rng = np.random.default_rng(9087)
pair = cointegrated_pair(450, rng, design)
assert np.allclose(pair.y - design.long_run_beta * pair.x, pair.latent_residual)
coefficient = ols_levels(pair.y, pair.x).slope
ecm = ecm_fit(pair.x, pair.y)
innovation = (np.diff(pair.y) - design.long_run_beta * np.diff(pair.x)
              - (design.adjustment_rho - 1) * pair.latent_residual[:-1])
assert np.isfinite(innovation).all()
assert abs(coefficient - design.long_run_beta) < 0.25
print({'estimated_longrun_slope': coefficient, **ecm,
       'true_adjustment_lambda': design.adjustment_rho - 1})

## 3. Distinguish observed-level DF from estimated-residual EG

The lag-zero observed-level Dickey–Fuller test with an intercept and the lag-zero two-step estimated-residual statistic without an intercept have **different null distributions**. These raw statistics are not accompanied by off-the-shelf ordinary Student-t p-values. See the main analysis for separate simulation-based 5% critical values under the exact stated null.

In [ ]:
df_stat = dickey_fuller_t(x, intercept=True)
eg_stat = engle_granger_t(x, y)
assert np.isfinite([df_stat, eg_stat]).all()
print({'df_level_statistic': df_stat, 'eg_estimated_residual_statistic': eg_stat})

## 4. Independently regenerate a reduced experiment

The canonical CSVs and figures in the repository were generated with the full settings. This final cell uses `quick=True` to verify all eight plotting paths and fifteen output tables from an isolated temporary output directory. Its quick-run numerical rates are **not** substituted for the canonical full-run rates in the article.

In [ ]:
with TemporaryDirectory(prefix='spurious-regression-notebook-') as tmpdir:
    output = Path(tmpdir)
    smoke = run_study(output, quick=True)
    figures = sorted((output / 'figures').glob('*.png'))
    tables = sorted((output / 'results').glob('*.csv'))
    assert len(figures) == 8
    assert len(tables) == 15
print(f'Generated {len(figures)} figures, {len(tables)} tables; ',
      f'naive levels rejection: {smoke["rw_levels_false_rejection"]:.3f}')